# Capítulo 6: Dados: Tipos, Dados Retangulares e pandas

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-elementos-de-dados-estruturados.html) | Elementos de Dados Estruturados |
| [6.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/02-dados-retangulares.html) | Dados Retangulares |
| [6.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/03-lendo-e-tipando-um-arquivo-real.html) | Lendo e Tipando um Arquivo Real |
| [6.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/04-limpando-e-transformando.html) | Limpando e Transformando |
| [6.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/05-agrupando-e-resumindo.html) | Agrupando e Resumindo |
| [6.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/06-da-tabela-para-o-modelo.html) | Da Tabela para o Modelo |

## Elementos de Dados Estruturados

Todo modelo dos capítulos anteriores recebeu números prontos: um vetor de entrada, um alvo, um gradiente que decrescia passo a passo. Dado bruto não chega assim. Chega em uma tabela, com uma coluna que conta pessoas, outra que mede uma taxa, outra que nomeia um lugar — e cada uma pede um tratamento diferente antes de qualquer conta começar. A primeira pergunta, antes de qualquer ajuste, é de que tipo é cada coluna.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

### A taxonomia dos dados

Dado estruturado se divide em duas famílias. **Numérico** é o que se mede: contínuo é qualquer valor dentro de um intervalo — uma taxa, um comprimento, um preço —, discreto é contagem, sempre inteiro — número de filhos, de vendas, de acessos. **Categórico** é o que se rotula: nominal não tem ordem — uma cor, a sigla de um estado, o nome de uma cidade —, ordinal tem ordem mas não tem distância — uma faixa "baixa/média/alta" diz que alta vem depois de média, mas não diz *quanto* maior —, e binário é o caso particular de só dois valores, do tipo "sim ou não".

> **🔷 Conceito**
>
> | Família | Subtipo | Tem ordem? | Tem distância? | Exemplo |
> |---|---|---|---|---|
> | Numérico | Contínuo | sim | sim | uma taxa, um preço |
> | Numérico | Discreto | sim | sim | uma contagem |
> | Categórico | Nominal | não | não | uma sigla, uma cor |
> | Categórico | Ordinal | sim | não | uma faixa "baixa/média/alta" |
> | Categórico | Binário | — | não | "sim"/"não" |
>
> A distância é o que separa numérico de ordinal: em ambos dá para dizer "isto vem depois daquilo", mas só no numérico faz sentido perguntar "quanto depois".

O tipo de uma coluna decide qual gráfico faz sentido para ela, qual estatística pode ser calculada sobre ela e como um programa consegue validar o que entra nela; tratar uma variável ordinal como se fosse numérica produz conclusão errada com aparência de rigor — a média entre "baixa" e "alta" devolve um número, só que um número que não significa nada, porque a distância entre as duas nunca foi definida.

### O que o `pandas` infere sozinho

Para ver a distinção aplicada a um caso concreto, considere as unidades federativas do Brasil, com população e taxa de homicídios:

In [ ]:
estados = pd.read_csv("dados/estados.csv")
estados.dtypes

O `pandas` acerta metade da tarefa sozinho, só olhando a forma dos valores: `Populacao` vira `int64` — é contagem, portanto numérico discreto —, e `Taxa.Homicidios` vira `float64` — é uma taxa, portanto numérico contínuo. `Estado` e `Sigla` viram `object`, o tipo genérico de texto que o `pandas` usa quando não sabe o que mais dizer sobre uma coluna: ele enxerga strings, não que "RO" e "AC" vêm de um conjunto fechado de rótulos possíveis. É inferência sobre a *forma* do valor, não sobre o que ele *significa* — só quem lê o dado sabe que `Sigla` é categórico nominal.

In [ ]:
len(estados), estados["Sigla"].nunique()

As 27 linhas trazem 27 siglas distintas — nenhuma se repete.

### Declarando o categórico

Dizer ao `pandas` o que já se sabe sobre a coluna é uma linha:

In [ ]:
antes = estados.memory_usage(deep=True)
estados["Sigla"] = estados["Sigla"].astype("category")
estados["Sigla"].cat.categories

Antes de acreditar que a conversão economiza alguma coisa, meça. Comparando o consumo de memória coluna a coluna, de antes para depois:

In [ ]:
depois = estados.memory_usage(deep=True)
comparacao = pd.DataFrame({"object": antes, "category": depois})
comparacao.loc["total"] = comparacao.sum()
comparacao["variação (%)"] = (
    (comparacao["category"] - comparacao["object"]) / comparacao["object"] * 100
).round(1)
comparacao

`Sigla` sobe de 1.377 para 2.476 bytes — quase 80% a mais —, e como nenhuma outra coluna muda, o `DataFrame` inteiro sobe de 3.694 para 4.793 bytes, quase 30% a mais. Com 27 valores todos distintos, não há repetição nenhuma para o `category` amortizar: o array de códigos mais o índice de categorias, por cima do que já existia, custam mais do que os ponteiros de texto do `object`. O ganho aqui é **semântico, não de memória** — e, sem repetição para amortizar, nem podia ser: o que muda é que operações que só fazem sentido sobre um conjunto fechado de rótulos passam a existir, como listar as categorias possíveis ou recusar um valor que não está entre elas. A economia de memória aparece quando poucos valores se repetem em muitas linhas — é o caso que a seção 6.3 mostra, ao tipar uma coluna de cidade.

### Ordem sem distância: o categórico ordinal

Nem todo categórico é nominal. Uma faixa de população tem ordem — "6 a 15 milhões" vem depois de "2 a 6 milhões" — mesmo sem ter distância: a faixa não diz o quanto maior. `pd.cut` corta um numérico em faixas categóricas, e `ordered=True` guarda essa ordem em vez de descartá-la:

In [ ]:
faixa = pd.cut(
    estados["Populacao"],
    bins=[0, 2_000_000, 6_000_000, 15_000_000, 50_000_000],
    labels=["até 2 milhões", "2 a 6 milhões", "6 a 15 milhões", "mais de 15 milhões"],
    ordered=True,
)
faixa.value_counts()

Dez estados ficam na faixa intermediária, de 2 a 6 milhões; só três passam de 15 milhões. Com a ordem declarada, comparar deixa de ser comparar texto e passa a ser comparar posição:

In [ ]:
(faixa < "6 a 15 milhões").sum()

Quinze estados ficam abaixo da faixa "6 a 15 milhões" — o `<` olha a posição de cada faixa na ordem que `ordered=True` fixou, não a ordem alfabética dos rótulos (que poria "6 a 15 milhões" antes de "até 2 milhões", e a conta sairia errada).

> **🟩 Exemplo**
>
> Pedir a mesma comparação a uma coluna nominal não tem resposta, e o `pandas` recusa a pergunta em vez de inventar uma:

In [ ]:
try:
    estados["Sigla"] < "SP"
except TypeError as erro:
    print(erro)

> Sem ordem declarada, "menor que" não está definido para `Sigla` — é a mesma pergunta que devolveria um número, e uma falsa sensação de precisão, se a coluna tivesse ficado como texto solto em vez de `category`.

### O tipo escolhe o gráfico

A mesma distinção decide o gráfico. Um categórico com um número por rótulo pede barras — a pergunta é "quanto vale cada um"; um numérico contínuo pede histograma — a pergunta é "como os valores se distribuem":

In [ ]:
# Figura: Um categórico (a unidade federativa) pede barras; um contínuo (a taxa de homicídios) pede histograma — o tipo escolhe o gráfico
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

top10 = estados.nlargest(10, "Populacao").sort_values("Populacao")
ax1.barh(top10["Estado"], top10["Populacao"] / 1e6)
ax1.set_xlabel("população (milhões)")
ax1.set_title("10 maiores populações")

ax2.hist(estados["Taxa.Homicidios"], bins=8)
ax2.set_xlabel("taxa de homicídios (por 100 mil hab.)")
ax2.set_ylabel("estados")
ax2.set_title("distribuição da taxa de homicídios")

plt.tight_layout()
plt.show()

`estados` ainda é só quatro colunas soltas, cada uma com o seu tipo. A próxima seção olha a estrutura que as mantém juntas — o `DataFrame` em si, a sua forma, o seu índice, e as duas formas de escolher linha e coluna dentro dele.

## Dados Retangulares

A seção anterior tratou `estados` como quatro colunas soltas, cada uma com o seu tipo — mas nenhuma coluna existe sozinha: são as linhas que as amarram, cada uma descrevendo a mesma unidade federativa em `Estado`, `Populacao`, `Taxa.Homicidios` e `Sigla`. É essa junção — colunas compartilhando linhas — que faz de `estados` uma tabela, e não uma pilha de quatro listas independentes.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

### A forma é uma escolha, não uma lei

`shape` mede essa junção direto:

In [ ]:
estados = pd.read_csv("dados/estados.csv")
estados.shape

Linhas são registros — cada uma, uma unidade federativa —, e colunas são variáveis — cada uma, uma medida ou um rótulo sobre ela. O par `(27, 4)` só quer dizer alguma coisa porque se sabe o que cada eixo representa: 27 estados, 4 variáveis por estado. Se as duas dimensões trocassem de lugar — 4 linhas, 27 colunas —, a tabela teria os mesmos números escritos, só que descrevendo outro problema: cada linha seria uma variável, e cada coluna, um estado.

`head()` mostra as primeiras linhas dessa junção:

In [ ]:
estados.head()

`info()` acrescenta o que `head()` não mostra: quantos valores não nulos há em cada coluna, e quanta memória a tabela ocupa.

In [ ]:
estados.info()

As quatro colunas têm 27 valores não nulos cada — nenhum furo nesta tabela —, e o total ocupa 996,0+ bytes; o sinal de mais avisa que essa é uma estimativa rasa, que conta menos que o real para as colunas de texto, como a seção anterior já tinha medido de outro jeito.

> **🔷 Conceito**
>
> Linhas são registros, colunas são variáveis. É essa convenção — não uma lei da natureza — que faz `(27, 4)` significar "27 estados, 4 variáveis" em vez de "27 variáveis, 4 estados". Trocar os eixos preservaria os números e destruiria o significado.

### Da posição ao rótulo: o índice

Toda tabela tem um índice, ainda que ele passe despercebido. Por padrão, o `pandas` numera as linhas pela posição em que chegaram:

In [ ]:
estados.index

0, 1, 2, e assim por diante — a mesma ordem do arquivo, sem relação com o conteúdo de nenhuma coluna. `set_index` troca esse índice posicional por um rótulo tirado dos próprios dados:

In [ ]:
por_sigla = estados.set_index("Sigla")
por_sigla.loc["SP"]

Com `Sigla` como índice, `.loc["SP"]` deixa de ser uma posição para virar uma busca por rótulo — ela devolve a linha de São Paulo, com população de 45.973.194 e taxa de 6,16 homicídios por 100 mil habitantes, sem que se precise saber em que posição do arquivo aquela linha estava. É uma busca que a posição sozinha nunca permitiria com sentido.

### `.loc` contra `.iloc`

Trocar o índice não apaga a posição — só deixa de ser o caminho padrão para chegar numa linha. `.iloc` continua contando por posição, `.loc` passa a contar por rótulo, e os dois convivem na mesma tabela:

In [ ]:
por_sigla.iloc[0]

`.iloc[0]` continua trazendo Rondônia — a primeira linha do arquivo original —, porque `.iloc` ignora o índice e enxerga só a posição. `.loc["SP"]`, feito na mesma tabela, nunca devolveria essa linha: São Paulo não está na posição 0, está no rótulo `"SP"`.

> **🟩 Exemplo**
>
> Na mesma tabela `por_sigla`, `.iloc[0]` devolve Rondônia e `.loc["SP"]` devolve São Paulo — duas linhas diferentes, ambas corretas, cada uma respondendo a uma pergunta diferente: "qual é a primeira linha?" contra "qual linha tem o rótulo SP?".

### Uma coluna, duas formas: `Series` e `DataFrame`

Selecionar uma coluna também tem duas respostas possíveis, e a sintaxe decide qual delas volta. Colchete simples devolve a coluna nua; colchete duplo devolve uma tabela de uma coluna só:

In [ ]:
print(type(estados["Populacao"]))
print(type(estados[["Populacao"]]))

In [ ]:
print(estados["Populacao"].shape)
print(estados[["Populacao"]].shape)

`estados["Populacao"]` devolve uma `Series` — um vetor de 27 valores com índice, formato `(27,)`. `estados[["Populacao"]]`, com colchete duplo, devolve um `DataFrame` — a mesma informação embrulhada numa tabela de uma coluna, formato `(27, 1)`. A diferença aparece toda vez que um método espera uma tabela e recebe um vetor, ou o contrário, e volta a importar mais adiante, na seção 6.6.

### O que a tabela não mostra

Olhando coluna a coluna, `estados` não deixa ver se população e violência caminham juntas. Isso pede um gráfico, não mais uma linha de código sobre uma coluna só:

In [ ]:
# Figura: População (escala log) contra taxa de homicídios: a correlação é fraca e negativa — tamanho, por si só, não decide violência
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(estados["Populacao"], estados["Taxa.Homicidios"])
ax.set_xscale("log")

destaques = ["RR", "AP", "PE", "MG", "SP"]
for _, linha in estados[estados["Sigla"].isin(destaques)].iterrows():
    ax.annotate(
        linha["Sigla"],
        (linha["Populacao"], linha["Taxa.Homicidios"]),
        xytext=(6, 4),
        textcoords="offset points",
    )

ax.set_xlabel("população (escala log)")
ax.set_ylabel("taxa de homicídios (por 100 mil hab.)")
plt.tight_layout()
plt.show()

A nuvem de pontos não desenha uma diagonal clara: em quase todo nível de população há um estado com taxa alta e outro com taxa baixa perto dele. Os dois estados de menor população ilustram isso de perto:

In [ ]:
estados.nsmallest(2, "Populacao")[["Sigla", "Populacao", "Taxa.Homicidios"]]

In [ ]:
estados.nlargest(2, "Populacao")[["Sigla", "Populacao", "Taxa.Homicidios"]]

Roraima e Amapá têm populações de 716.793 e 802.837 — diferença de menos de 100 mil habitantes —, e taxas de 23,58 e 32,01: quase vizinhos num eixo, distantes no outro. No outro extremo, São Paulo (45.973.194 habitantes) e Minas Gerais (21.322.691), as duas maiores populações da tabela, têm taxas de 6,16 e 12,63 — ambas entre as mais baixas. E a maior taxa de todas não é de um estado pequeno nem do maior:

In [ ]:
estados.nlargest(1, "Taxa.Homicidios")[["Sigla", "Populacao", "Taxa.Homicidios"]]

Pernambuco tem a maior taxa da tabela, 36,78 — e não é um estado pequeno:

In [ ]:
print("mediana de Populacao:", estados["Populacao"].median())
print(
    "posição de PE por população (1 = maior):",
    estados.sort_values("Populacao", ascending=False)["Sigla"].tolist().index("PE") + 1,
    "de",
    len(estados),
)

Pernambuco é o sétimo mais populoso das 27 unidades, com população acima do dobro da mediana. A correlação entre as duas colunas dá um primeiro número para o que o olho já viu:

In [ ]:
round(estados["Populacao"].corr(estados["Taxa.Homicidios"]), 2)

-0,41 é uma correlação fraca. Tirando os dois estados que a puxam no topo da população:

In [ ]:
sem_sp_mg = estados[~estados["Sigla"].isin(["SP", "MG"])]
round(sem_sp_mg["Populacao"].corr(sem_sp_mg["Taxa.Homicidios"]), 2)

ela cai de -0,41 para -0,02 — praticamente zero. São Paulo e Minas Gerais sozinhos respondem pela tendência inteira; tirados os dois, população e taxa não guardam relação nenhuma entre si no resto da distribuição. Separar os estados por região, o que agrupar e resumir faz na seção 6.5, é o próximo passo para procurar o que de fato explica a variação — população isolada não é essa resposta.

A tabela usada até aqui não tem um valor faltante sequer. A próxima seção troca esse arquivo limpo por um que chega direto de uma fonte real, com furos e tipos que o `pandas` não resolve sozinho.

## Lendo e Tipando um Arquivo Real

A seção anterior fechou avisando que `estados` não tinha um valor faltante sequer, e prometeu trocar esse arquivo limpo por um que chega direto de uma fonte real. É o que este arquivo faz: um conjunto de anúncios de aluguel de imóveis em cinco cidades brasileiras, lido sem nenhuma limpeza prévia — exatamente como ele chegaria de um raspador ou de uma exportação de planilha.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

### Uma primeira leitura

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv")
alugueis.shape

10.692 imóveis, 13 colunas. `head()` mostra o que cada linha descreve:

In [ ]:
alugueis.head()

Cada linha é um anúncio: a cidade, a área em metros quadrados, quantos quartos, banheiros e vagas de garagem, o andar, se aceita animal, se vem mobiliado, e cinco valores em reais — condomínio, aluguel, IPTU, seguro-incêndio e o total.

### Uma coluna fora do padrão

Antes de qualquer conta, `dtypes` — o mesmo primeiro passo da seção 6.1:

In [ ]:
alugueis.dtypes

Doze colunas vieram como o esperado: nove numéricas (`area_m2` a `total`, todas contagens ou valores em reais) e três como `object` porque são texto de verdade — `cidade` é um nome de cidade, `aceita_animal` e `mobiliado` são "sim" ou "não". `andar` é a única que destoa: representa um número — o andar do imóvel —, e mesmo assim veio como `object`, junto dos textos.

### O estrago, antes da causa

Um número de andar deveria admitir média. Tentar calcular uma mostra o problema na prática, não em teoria:

In [ ]:
try:
    alugueis["andar"].mean()
except TypeError as erro:
    print("TypeError:", str(erro)[:90], "...")

`mean()` nem chega a rodar: `object` é o tipo genérico que o `pandas` usa para texto, e texto não tem média. A mensagem completa, se fosse impressa inteira, traria a coluna inteira concatenada num único bloco de mais de 10 mil caracteres — por isso o truncamento acima, e não por economia de espaço.

### Onde está o problema

`unique()` mostra os primeiros valores distintos da coluna:

In [ ]:
alugueis["andar"].unique()[:10]

Entre `'7'`, `'20'`, `'6'` e outros números escritos como texto, aparece um `'-'`. Contando quantas linhas trazem exatamente esse valor:

In [ ]:
tracos = (alugueis["andar"] == "-").sum()
tracos, round(100 * tracos / len(alugueis))

2.461 das 10.692 linhas — 23% do arquivo — têm `"-"` em vez de um número de andar. Basta uma linha assim para que a coluna inteira vire `object`: o `pandas` decide o tipo de uma coluna olhando *todos* os valores dela, e uma coluna não pode ser meio `int64`, meio texto.

### Por que o `pandas` não pegou sozinho

O `read_csv` já reconhece um conjunto de marcadores de valor ausente — `NaN`, `NA`, `null`, `n/a`, campo vazio, entre outros — e os converte direto em nulo, sem que ninguém peça. `"-"` não está nessa lista. Para o `pandas`, `"-"` é um caractere como outro qualquer, e uma coluna com um único valor não numérico entre milhares de números vira `object` inteira, sem aviso.

> **🔷 Conceito**
>
> A inferência de tipo do `pandas` é uma **heurística sobre os caracteres do arquivo**, não um contrato sobre o que a coluna significa. Ela acerta quando todo valor é uma forma reconhecível de número, texto ou marcador de nulo já catalogado — e erra silenciosamente quando um valor foge desse catálogo, mesmo sendo óbvio para quem lê a coluna que `"-"` ali significa "sem informação". Conferir `dtypes` logo depois de ler o arquivo é o que transforma esse silêncio em alguma coisa visível antes de qualquer conta.

### Dois consertos, para duas situações

Quando se sabe qual é o marcador, `na_values` ensina o `read_csv` a reconhecê-lo já na leitura:

In [ ]:
alugueis_tipado = pd.read_csv("dados/alugueis.csv", na_values=["-"])
alugueis_tipado["andar"].dtype, alugueis_tipado["andar"].isna().sum(), round(alugueis_tipado["andar"].mean(), 2)

Com `"-"` na lista de nulos, `andar` vira `float64` — os 2.461 traços viram `NaN`, e a média dos demais 8.231 valores sai 6,58.

Quando não se sabe de antemão qual marcador vai aparecer, `pd.to_numeric` com `errors="coerce"` faz o trabalho na direção contrária: converte o que consegue e transforma **tudo o que não é número** em nulo, seja `"-"`, seja qualquer outro caractere estranho que o arquivo trouxer.

In [ ]:
andar_coerced = pd.to_numeric(alugueis["andar"], errors="coerce")
andar_coerced.dtype, andar_coerced.isna().sum()

O resultado bate com o de `na_values` neste arquivo — os mesmos 2.461 nulos —, porque `"-"` é o único valor fora do padrão numérico. Mas `to_numeric` é mais forte e mais perigoso do que parece: ele converte em silêncio qualquer valor que não reconheça, inclusive um erro de digitação que `na_values=["-"]` deixaria passar direto como texto visível, pronto para ser notado. Usar `na_values` é dizer "eu sei o que este marcador significa"; usar `to_numeric(errors="coerce")` é dizer "eu não sei o que vou encontrar, e prefiro perder o que não for número a travar a conversão inteira".

### Uma coluna em que `category` compensa

A seção 6.1 converteu `Sigla` em `category` e mediu que o consumo de memória *piorava* — 27 valores, todos distintos, sem repetição para amortizar. `cidade` pode ser o caso oposto, se poucos valores se repetirem por muitas linhas. Antes de converter, vale medir o consumo de memória como está:

In [ ]:
antes_cidade = alugueis["cidade"].memory_usage(deep=True)
antes_cidade

730.981 bytes para uma coluna de texto. Quantos valores distintos existem por trás desse número:

In [ ]:
alugueis["cidade"].nunique()

Só cinco cidades, repetidas pelas 10.692 linhas — a proporção entre valores distintos e linhas é o inverso exato do caso de `Sigla`. Convertendo e medindo de novo:

In [ ]:
depois_cidade = alugueis["cidade"].astype("category").memory_usage(deep=True)
antes_cidade, depois_cidade, round(depois_cidade / antes_cidade * 100)

De 730.981 bytes como `object` para 11.325 bytes como `category` — 2% do tamanho original. Aqui a repetição é o que faz o `category` compensar: em vez de 10.692 ponteiros de texto, a coluna guarda cinco categorias e um array de códigos pequenos apontando para elas. É o mesmo mecanismo da 6.1, com o resultado invertido porque a proporção entre valores distintos e linhas também está invertida.

### A cauda que o aluguel esconde

Resolvido o problema de `andar`, vale olhar o que a distribuição do próprio aluguel revela:

In [ ]:
# Figura: Distribuição do aluguel: a maioria dos imóveis se concentra abaixo de 10 mil reais, mas a cauda longa chega a 45 mil
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(alugueis["aluguel"], bins=60)
ax.set_xlabel("aluguel (R$)")
ax.set_ylabel("imóveis")
plt.tight_layout()
plt.show()

In [ ]:
alugueis["aluguel"].median(), alugueis["aluguel"].max()

A mediana fica em R$ 2.661, e a barra mais alta do histograma mora perto dela — mas a cauda à direita se estica até R$ 45.000, um único imóvel bem separado do resto. Uma coluna corretamente tipada não garante uma distribuição bem comportada: o `aluguel` já é `int64` desde a primeira leitura, e ainda assim traz um outlier que qualquer estatística resumida esconderia. É esse tipo de problema — não mais o tipo da coluna, mas o formato dos valores dentro dela — que a próxima seção enfrenta.

## Limpando e Transformando

A seção anterior fechou com `aluguel` já corretamente tipado como `int64` e, mesmo assim, escondendo um imóvel isolado a R$ 45.000 — um problema que não é mais de tipo, mas do que os valores contêm. `andar` é o próximo caso, e não é bem o mesmo problema: o marcador `"-"` já virou nulo com `na_values`, então a coluna está tipada; o que falta decidir é o que fazer com esses nulos, com as linhas repetidas que ainda não foram olhadas, e com os valores extremos que `describe()` revela.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"])
alugueis.shape

### Onde estão os nulos

Com `andar` já tipado, `isna().sum()` mostra o que sobrou:

In [ ]:
alugueis.isna().sum()[lambda s: s > 0]

Só `andar`, com 2.461 nulos — o mesmo número que a seção anterior contou como traços. Nenhuma outra coluna do arquivo tem valor faltando.

### Nulo não é uma coisa só

O reflexo depois de ver uma coluna com nulo é `dropna()`. Antes de aplicá-lo, vale perguntar o que esse nulo está dizendo — e a própria tabela responde, se for comparada consigo mesma. `condominio` e `area_m2` entre as linhas em que `andar` é nulo e as linhas em que não é:

In [ ]:
alugueis.groupby(alugueis["andar"].isna())[["condominio", "area_m2"]].median()

Nas linhas com `andar` nulo, a mediana de `condominio` é 0 e a de `area_m2` é 190 m². Nas linhas com `andar` preenchido, a mediana de `condominio` sobe para 750 e a de `area_m2` cai para 80 m². A proporção de imóveis sem condomínio nenhum aprofunda o mesmo padrão:

In [ ]:
mascara = alugueis["andar"].isna()
zero_nulo = round(100 * (alugueis.loc[mascara, "condominio"] == 0).sum() / mascara.sum(), 1)
zero_preenchido = round(100 * (alugueis.loc[~mascara, "condominio"] == 0).sum() / (~mascara).sum(), 1)
zero_nulo, zero_preenchido

84,4% das linhas com `andar` nulo têm `condominio` igual a zero, contra 3,6% das linhas com `andar` preenchido. Um imóvel sem condomínio e maior que a mediana é o perfil de uma casa, não de um apartamento. O arquivo não traz uma coluna de tipo de imóvel — o que ele traz é esse padrão, repetido em duas variáveis diferentes, e é ele, não uma afirmação de cabeça, que sustenta a leitura de que `andar` nulo aqui não é o dado faltando: é o imóvel não ter andar, porque é uma casa. Vale a ressalva: isso é evidência forte, não prova — ninguém confere linha por linha se cada uma dessas 2.461 é mesmo uma casa. É a mesma leitura que se faz de qualquer dado sem rótulo direto: somar sinais indiretos até que só uma explicação os explique todos ao mesmo tempo.

Isso muda a decisão. Descartar essas 2.461 linhas jogaria fora informação que não estava ausente — o imóvel existe, o aluguel existe, só o andar não faz sentido para ele. Preencher com 0 é a escolha coerente com o que o dado significa: 0 é o andar de quem não tem andar, o térreo, que é exatamente o que uma casa é.

In [ ]:
sem_nulos = alugueis.dropna()
com_zero = alugueis.fillna({"andar": 0})
alugueis.shape, sem_nulos.shape, com_zero.shape

In [ ]:
round(100 * (alugueis.shape[0] - sem_nulos.shape[0]) / alugueis.shape[0])

`dropna()` sai de 10.692 para 8.231 linhas — 23% da tabela, por uma coluna em que o nulo nunca foi ausência. `fillna({"andar": 0})` mantém as 10.692 e escreve, em número, a mesma coisa que o traço já dizia em texto. É essa a versão que fica:

In [ ]:
alugueis["andar"] = alugueis["andar"].fillna(0)
alugueis["andar"].isna().sum()

> **🔷 Conceito**
>
> `dropna()` e `fillna()` não são um mecânico melhor que o outro — são duas respostas a perguntas diferentes sobre o que o nulo significa. `dropna()` presume que a linha está incompleta e que o certo é não contar com ela. `fillna(valor)` presume que o nulo é, ele mesmo, informação, e escolhe o número que a representa. A segunda presunção só se sustenta quando alguém conhece o domínio dos dados o bastante para saber que "sem andar" e "casa" são a mesma coisa — o `pandas` não tem como adivinhar isso sozinho, e a linha `fillna({"andar": 0})` é, por si só, muda sobre o motivo.

### Duplicatas

In [ ]:
alugueis.duplicated().sum()

358 linhas repetem, em todas as 13 colunas, uma linha que já apareceu antes. Contando também as primeiras ocorrências de cada grupo repetido, e como esses grupos se distribuem por tamanho:

In [ ]:
repetidos = alugueis[alugueis.duplicated(keep=False)]
tamanhos = repetidos.groupby(list(alugueis.columns), dropna=False).size()

print("linhas repetidas:", len(repetidos))
print("grupos:", len(tamanhos))
print("pares:", (tamanhos == 2).sum())
print("de três ou mais:", (tamanhos >= 3).sum(), "grupos,", tamanhos[tamanhos >= 3].sum(), "linhas")
print("maior grupo:", tamanhos.max())

Dos 246 grupos, 206 são pares. Os outros 40 têm de três a 22 cópias, e são esses 40 grupos, sozinhos, que respondem pelas 192 linhas que sobram das 604. O maior — 22 anúncios idênticos em todas as 13 colunas — leva ao extremo a mesma pergunta que qualquer duplicata levanta: é um imóvel (ou um anúncio) republicado vinte e duas vezes, ou vinte e dois apartamentos realmente iguais? A tabela não traz o que decidiria entre as duas. Um exemplo, de um grupo menor:

In [ ]:
repetidos.sort_values(["cidade", "area_m2"]).head(3)

Três anúncios de Belo Horizonte, 15 m², um quarto, idênticos até o último real de condomínio e seguro-incêndio.

> **🟩 Exemplo**
>
> Uma linha repetida pode ser duas coisas bem diferentes: o mesmo imóvel anunciado mais de uma vez — pela imobiliária e pelo dono, ou reenviado depois de expirar —, ou dois apartamentos realmente iguais, no mesmo prédio, com a mesma planta e o mesmo condomínio. A tabela não guarda um identificador de anúncio nem um endereço, e sem isso não há como distinguir as duas situações a partir do que está nela. Remover as duplicatas presumiria a primeira explicação; mantê-las presumiria a segunda. Nenhuma das duas é sustentada pelo dado disponível — e é por isso que esta seção não decide: `alugueis` segue com as 10.692 linhas, duplicadas incluídas.

### Outliers

In [ ]:
alugueis[["area_m2", "condominio"]].describe()

Em `area_m2`, a mediana é 90 m² contra um máximo de 46.335 m². Em `condominio`, a mediana é R$ 560 contra um máximo de R$ 1.117.000. Como razão entre máximo e mediana:

In [ ]:
razao_area = round(alugueis["area_m2"].max() / alugueis["area_m2"].median())
razao_condominio = round(alugueis["condominio"].max() / alugueis["condominio"].median())
razao_area, razao_condominio

O máximo de `area_m2` é 515 vezes a mediana da própria coluna; o de `condominio`, 1.995 vezes. E o topo de `area_m2` não é um ponto isolado — os cinco maiores:

In [ ]:
alugueis["area_m2"].sort_values(ascending=False).head(5).tolist()

46.335, 24.606 e 12.732 m² são os três imóveis acima de dez mil metros quadrados. O quarto e o quinto maior já caem bem abaixo disso — e empatam entre si, os dois em exatos 2.000 m². Em `condominio`, o maior valor não é um ponto isolado no outro sentido: ele se repete.

In [ ]:
alugueis[alugueis["condominio"] == alugueis["condominio"].max()]

As duas linhas com condomínio de R$ 1.117.000 são o mesmo par que a seção de duplicatas já contou: mesma cidade, mesma área, mesmo aluguel, tudo igual — um imóvel (ou um anúncio) registrado duas vezes, não dois imóveis diferentes que coincidem em ter condomínio alto.

In [ ]:
p99_area = round(alugueis["area_m2"].quantile(0.99))
p99_area

Noventa e nove por cento dos imóveis têm até 650 m².

In [ ]:
# Figura: Área dos imóveis: o boxplot da esquerda comprime tudo perto de zero por causa dos poucos imóveis muito grandes; cortando o eixo no percentil 99, a distribuição do restante aparece
fig, axs = plt.subplots(1, 2, figsize=(8, 5))

axs[0].boxplot(alugueis["area_m2"])
axs[0].set_title("todos os imóveis")
axs[0].set_ylabel("área (m²)")

axs[1].boxplot(alugueis["area_m2"])
axs[1].set_ylim(0, p99_area)
axs[1].set_title(f"eixo cortado no percentil 99 ({p99_area} m²)")

plt.tight_layout()
plt.show()

À esquerda, a caixa inteira do boxplot — onde mora metade dos imóveis — vira uma linha reta colada no zero: os imóveis de 46.335, 24.606 e 12.732 m² esticam o eixo inteiro para caber. À direita, com o eixo cortado em 650 m², a caixa aparece, e são esses mesmos imóveis — e outros acima do corte — que ficam fora do que o gráfico mostra.

Nenhuma dessas linhas sai da tabela. O imóvel de 46.335 m² e o condomínio de R$ 1.117.000 continuam em `alugueis` até o fim desta seção e da próxima: são o motivo pelo qual a seção seguinte, ao resumir valores por cidade, prefere a mediana à média — um valor deste tamanho desloca uma média inteira, e a mediana não se importa com ele.

### Uma coluna derivada

In [ ]:
alugueis["aluguel_por_m2"] = alugueis["aluguel"] / alugueis["area_m2"]
alugueis["aluguel_por_m2"].describe()

`aluguel_por_m2` divide, linha a linha, o aluguel pela área — uma conta que o arquivo não trazia pronta, e que agora é só mais uma coluna de `alugueis`. Ela também mostra o efeito de um outlier de área por um ângulo diferente: qual é o imóvel do menor valor da coluna, e de onde vem esse valor:

In [ ]:
linha_minima = alugueis.loc[alugueis["aluguel_por_m2"].idxmin()]
round(linha_minima["aluguel_por_m2"], 2), round(linha_minima["area_m2"])

R$ 0,13 por metro quadrado — não porque o aluguel seja baixo, mas porque a área é 12.732 m², um dos três imóveis extremos já vistos, grande o bastante para que qualquer aluguel divida em quase nada. É essa coluna que a próxima seção compara entre as cinco cidades.

## Agrupando e Resumindo

A seção anterior fechou com `alugueis` inteira, as 10.692 linhas mantidas, e uma coluna nova, `aluguel_por_m2`, dividindo o aluguel pela área de cada imóvel. O que falta é olhar para essa tabela não imóvel a imóvel, mas cidade a cidade: quanto custa alugar em cada uma, o que muda de uma para outra, e como trazer para dentro dela o que outras tabelas sabem sobre essas cinco cidades.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"])
alugueis["andar"] = alugueis["andar"].fillna(0)
alugueis["aluguel_por_m2"] = alugueis["aluguel"] / alugueis["area_m2"]
alugueis.shape

### Duas formas de resumir, dois tipos de coluna

`describe()` resume o que é contínuo — média, desvio, quartis:

In [ ]:
alugueis.describe()

`cidade` não é uma dessas colunas. É categórica, e a pergunta que cabe a ela não é "qual é a média", é "quantos de cada":

In [ ]:
alugueis["cidade"].value_counts()

In [ ]:
round(100 * alugueis["cidade"].value_counts()["São Paulo"] / len(alugueis), 1)

São Paulo sozinha responde por 55,1% dos 10.692 imóveis — mais da metade da tabela é de uma cidade só. É a mesma distinção que a seção 6.1 fez entre dado quantitativo e qualitativo: `describe()` não teria o que fazer com `cidade`, e `value_counts()` não teria o que fazer com `aluguel`.

### Mediana contra média, cidade por cidade

A seção anterior deixou de propósito o imóvel de R$ 45.000 mensais e o condomínio de R$ 1.117.000 dentro de `alugueis`, porque descartá-los seria decidir por um lado sem prova. O preço desse adiamento aparece agora: qualquer média calculada sobre `aluguel` carrega esses valores junto.

In [ ]:
mediana_media = alugueis.groupby("cidade")["aluguel"].agg(["median", "mean"]).round(2)
mediana_media

In [ ]:
(mediana_media["mean"] > mediana_media["median"]).all()

Nas cinco cidades, sem exceção, a média fica acima da mediana. Em São Paulo a distância é maior em valor absoluto — mediana de R$ 3.400 contra média de R$ 4.652,79:

In [ ]:
round(mediana_media.loc["São Paulo", "mean"] - mediana_media.loc["São Paulo", "median"], 2)

R$ 1.252,79 de diferença, numa cidade só. É por isso que o resumo por cidade, daqui em diante, usa a mediana: ela não se importa com os poucos imóveis que puxam a média para cima.

### Uma tabela por cidade

`agg` com várias funções e várias colunas de uma vez produz, numa chamada, o que seria preciso montar coluna a coluna: contagem, mediana do aluguel e mediana do aluguel por metro quadrado, para cada cidade.

In [ ]:
resultado = alugueis.groupby("cidade").agg(
    contagem=("aluguel", "count"),
    aluguel_mediano=("aluguel", "median"),
    aluguel_por_m2_mediano=("aluguel_por_m2", "median"),
).round(2).sort_values("aluguel_por_m2_mediano")
resultado

A coluna `contagem` é o mesmo número que `value_counts()` já tinha mostrado, agora ao lado do preço. Ordenada pelo aluguel por metro quadrado, a tabela por si só responde onde o metro quadrado é mais barato e onde é mais caro:

In [ ]:
# Figura: Mediana do aluguel por m² em cada cidade, da mais barata à mais cara
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(resultado.index, resultado["aluguel_por_m2_mediano"])
ax.set_xlabel("aluguel por m² (R$, mediana)")
plt.tight_layout()
plt.show()

### Juntando outras tabelas: o `merge` que não pode multiplicar linha

`alugueis` não sabe em que estado cada cidade fica, nem quantas pessoas vivem lá — isso está em `dados/cidades.csv` e `dados/estados.csv`. Um `merge` costura essas tabelas pela coluna que elas têm em comum, e é exatamente aí que mora um erro fácil de cometer e fácil de não perceber: se a chave do `merge` se repetir do lado que está sendo trazido, cada repetição gera uma cópia da linha original. A tabela cresce, e nada na tela avisa — o `merge` roda, devolve algo, e o algo é silenciosamente errado. A defesa é olhar o `shape` antes e depois de todo `merge`, sempre.

In [ ]:
cidades = pd.read_csv("dados/cidades.csv")
shape_antes = alugueis.shape
alugueis = alugueis.merge(cidades, on="cidade")
shape_antes, alugueis.shape

De (10692, 14) para (10692, 16): as mesmas 10.692 linhas, com `Sigla` e `regiao` a mais. `cidades` tem uma linha por cidade — a chave não se repete do lado trazido, e por isso o `merge` não multiplicou nada.

`dados/estados.csv` tem mais colunas que isso — nome do estado, taxa de homicídios —, mas só a população interessa a uma tabela de aluguéis, e é só ela que entra no `merge`:

In [ ]:
estados = pd.read_csv("dados/estados.csv")[["Sigla", "Populacao"]]
shape_antes = alugueis.shape
alugueis = alugueis.merge(estados, on="Sigla")
shape_antes, alugueis.shape

De novo, a contagem de linhas não muda: (10692, 16) vira (10692, 17), só com `Populacao` a mais. `estados` também tem uma linha por sigla, então cada linha de `alugueis` encontra exatamente uma correspondência do outro lado.

Nenhum dos dois `merge` multiplicou linha — mas isso não quer dizer que a chave usada identifique a cidade. `Sigla` identifica o **estado**, e duas das cinco cidades, São Paulo e Campinas, dividem o mesmo estado:

In [ ]:
alugueis.loc[alugueis["cidade"].isin(["São Paulo", "Campinas"]), ["cidade", "Sigla", "Populacao"]].drop_duplicates()

As duas linhas trazem `Populacao` igual — 45.973.194 —, porque é a população do estado de São Paulo inteiro, não da cidade. Um `merge` correto pela contagem de linhas ainda pode juntar a coisa errada com a coisa certa: `Populacao`, a partir daqui, descreve o estado onde o imóvel está, não o município.

> **🔷 Conceito**
>
> Um `merge` de muitos-para-um — muitas linhas de um lado, uma correspondência só do outro — não muda o número de linhas: cada linha do lado "muitos" encontra exatamente uma parceira e sai dali com as colunas novas coladas. Se a contagem de linhas depois do `merge` for maior do que antes, a chave se repete no lado que deveria ser único, e cada repetição multiplicou uma linha que já existia. É por isso que conferir o `shape` antes e depois não é cautela de sobra: é o único jeito de saber, olhando, se o `merge` fez o que parecia fazer.

> **🟩 Exemplo**
>
> São Paulo e Campinas mostram o outro lado do mesmo cuidado: o `merge` pela `Sigla` não multiplicou nenhuma linha — e mesmo assim juntou uma informação de estado a uma tabela de cidades, fazendo duas cidades bem diferentes em tamanho carregarem o mesmo número de população. A contagem de linhas certa não garante que a coluna trazida signifique o que parece significar para cada linha.

`alugueis` termina esta seção com 10.692 linhas e 17 colunas — a mesma tabela do início, mais o que `cidades.csv` e `estados.csv` acrescentaram. É essa tabela que a próxima seção decide como entregar a um modelo.

## Da Tabela para o Modelo

A seção anterior fechou `alugueis` com mais colunas do que abriu — cidade ganhou estado e população, e o aluguel ganhou uma versão por metro quadrado, sem que nenhum `merge` multiplicasse uma linha sequer. Nenhuma dessas colunas resolve a pergunta que fecha o capítulo, porque a pergunta não pede mais colunas: pede uma resposta. "Quanto deveria custar o aluguel deste imóvel?" é o que decide, de toda a tabela, o que vira o que um modelo tenta prever e o que vira o que ele recebe para prever.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

Como cada `.qmd` roda no seu próprio kernel, esta seção lê `dados/alugueis.csv` de novo e leva adiante só as duas decisões que a seção anterior tomou e disse que ficam: `"-"` vira nulo em `andar` (seção 6.3), e esse nulo vira 0 (seção 6.4). As colunas que os dois `merge` acrescentaram não entram nesta escolha — nenhuma pergunta abaixo depende de estado ou de população.

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"])
alugueis["andar"] = alugueis["andar"].fillna(0)
alugueis.shape

### A pergunta decide a tabela

O que a pergunta pede é `aluguel`: essa coluna sai da tabela e vira `y`, o alvo. O resto — o que se sabe do imóvel antes de saber o aluguel — vira `X`, os preditores. Por ora, só os numéricos que já chegam prontos: área, quartos, banheiros, vagas.

A seção 6.2 já mostrou que colchete simples devolve uma `Series` e colchete duplo devolve um `DataFrame`. É a mesma sintaxe que separa `y` de `X` aqui — só que agora `X` tem quatro colunas, não uma:

In [ ]:
y = alugueis["aluguel"]
X = alugueis[["area_m2", "quartos", "banheiros", "vagas"]]
print(type(y))
print(type(X))

In [ ]:
print(y.shape)
print(X.shape)

`y` é uma `Series` de formato `(10692,)`; `X` é um `DataFrame` de formato `(10692, 4)`. Cada linha das duas tabelas ainda descreve o mesmo imóvel — é essa correspondência linha a linha que faz `X` e `y` andarem juntos daqui em diante.

### O modelo não sabe o que é "cidade"

`cidade` é o que a seção 6.1 chamaria de categórico nominal: cinco rótulos, sem ordem nenhuma entre eles. Um modelo, porém, só sabe multiplicar e somar número — e a tradução mais óbvia de texto para número, trocar cada cidade por um inteiro (São Paulo = 1, Rio de Janeiro = 2, Campinas = 3, e assim por diante), inventaria exatamente o que `cidade` não tem: uma ordem entre as cinco, e uma distância entre elas — o modelo passaria a "ler" que Campinas está duas vezes mais longe de São Paulo do que o Rio está, uma afirmação sem sentido nenhum sobre nomes de cidade.

`pd.get_dummies` evita essa invenção: em vez de uma coluna com números inventados, cria uma coluna por categoria, cada uma com 0 ou 1.

In [ ]:
dummies_cidade = pd.get_dummies(alugueis["cidade"])
dummies_cidade.head(3)

In [ ]:
list(dummies_cidade.columns)

Cinco colunas, uma por cidade, em ordem alfabética. Nenhuma delas carrega mais peso que outra — e cada linha marca exatamente uma cidade como verdadeira:

In [ ]:
dummies_cidade.sum(axis=1).unique()

Toda linha soma 1: nenhum imóvel fica sem cidade, e nenhum marca duas ao mesmo tempo. Essas cinco colunas entram em `X` no lugar do texto:

In [ ]:
X = pd.concat([X, dummies_cidade], axis=1)
X.shape

`X` passa de quatro para nove colunas — as quatro numéricas mais as cinco 0/1 que substituem `cidade`, sem nenhuma ordem que a coluna original não tinha.

### Uma coluna que não pode entrar: o vazamento

A tabela ainda traz `total`, que devia ser a soma de `aluguel`, `condominio`, `iptu` e `seguro_incendio`. Conferindo:

In [ ]:
soma = alugueis["aluguel"] + alugueis["condominio"] + alugueis["iptu"] + alugueis["seguro_incendio"]
diferenca = soma - alugueis["total"]
(diferenca == 0).sum(), len(alugueis)

Em 9.347 das 10.692 linhas, a soma bate com `total` exatamente. Nas outras:

In [ ]:
divergentes = diferenca[diferenca != 0]
len(divergentes), divergentes.median(), divergentes.quantile(0.75), divergentes.min()

1.345 linhas divergem, e por pouco: a mediana da diferença é -1, o percentil 75 é 2, e o pior caso chega a -379 — a tabela não traz o que causou essas 1.345 exceções, só que a maioria bate exatamente e as demais divergem por um valor pequeno perto de zero, não por uma fração do total.

Uma coluna que é, para a imensa maioria das linhas, a soma que contém `aluguel` como parcela entrega a resposta a quem for prever `aluguel` — é essa a definição de **vazamento**, e é por isso que `total` não pode entrar em `X`. O instinto para confirmar um vazamento é medir a correlação entre a coluna suspeita e o alvo:

In [ ]:
round(alugueis["total"].corr(alugueis["aluguel"]), 4)

0,2645. Uma correlação fraca — para uma coluna que, na prática, contém `aluguel` dentro de si. Colocando `total` ao lado dos outros números da tabela, inclusive `condominio`, que compõe `total` do mesmo jeito:

In [ ]:
colunas_numericas = ["area_m2", "quartos", "banheiros", "vagas", "condominio", "total", "aluguel"]
correlacoes = alugueis[colunas_numericas].corr()
correlacoes.round(2)

In [ ]:
# Figura: Correlação entre os preditores numéricos, condominio, total e o alvo aluguel: banheiros se aproxima mais de aluguel do que total, mesmo total sendo, na prática, a soma que inclui aluguel
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(correlacoes, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(colunas_numericas)))
ax.set_xticklabels(colunas_numericas, rotation=45, ha="right")
ax.set_yticks(range(len(colunas_numericas)))
ax.set_yticklabels(colunas_numericas)
ax.grid(False)
for i in range(len(colunas_numericas)):
    for j in range(len(colunas_numericas)):
        valor = correlacoes.iloc[i, j]
        cor_texto = "white" if valor > 0.55 else "#373A3C"
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", color=cor_texto, fontsize=8)
fig.colorbar(im, ax=ax, label="correlação")
plt.tight_layout()
plt.show()

`banheiros` correlaciona 0,67 com `aluguel`, `vagas` 0,58, `quartos` 0,54 — todas mais altas que os 0,2645 de `total`, e nenhuma delas tem uma fórmula que inclua `aluguel`. Se a regra para achar vazamento fosse "procure a coluna de correlação mais alta com o alvo", `total` passaria despercebida, e `banheiros` chamaria a atenção por um motivo que não tem nada a ver com vazar resposta nenhuma.

O que explica os 0,2645 está na própria matriz: `total` correlaciona 0,96 com `condominio`, não com `aluguel`. `condominio` é a mesma coluna cujo outlier a seção anterior mostrou e manteve dentro de `alugueis` — e é ele quem decide de onde `total` puxa a sua variação. Removendo só os imóveis com o condomínio mais extremo:

In [ ]:
limite = alugueis["condominio"].quantile(0.999)
sem_outlier = alugueis[alugueis["condominio"] < limite]
len(alugueis) - len(sem_outlier), round(sem_outlier["total"].corr(sem_outlier["aluguel"]), 4)

Tirando 11 imóveis de 10.692 — o 0,1% de maior `condominio`, o mesmo extremo da seção anterior —, a correlação entre `total` e `aluguel` sobe de 0,2645 para 0,793. Nenhuma fórmula mudou: onze linhas, sozinhas, bastam para que a escala de `condominio` domine a variação de `total` e esconda, da correlação, o vínculo que a definição de `total` garante em quase toda a tabela.

> **🔷 Conceito**
>
> Vazamento não é uma propriedade da correlação — é uma propriedade de como a coluna foi construída. `total` vaza a resposta porque a fórmula que a gera inclui `aluguel` como parcela em 9.347 das 10.692 linhas, e isso continua verdade tanto com correlação de 0,2645 quanto de 0,793, com o outlier de `condominio` dentro da tabela ou fora dela. Medir a correlação com o alvo é um jeito de procurar vazamento — não o único, e este caso mostra como ele pode falhar: a coluna mais perigosa da tabela nem aparece entre as mais correlacionadas com o que se quer prever.

In [ ]:
"total" in X.columns

`total` segue fora de `X` — não porque a correlação avise, mas porque a definição da coluna é o motivo, e essa definição não muda de uma linha para outra. O que fazer com vazamento além de excluir a coluna volta mais adiante no material.

### `X` e `y`, prontos

In [ ]:
print(X.shape)
print(y.shape)

`X` chega com 10.692 linhas e nove colunas — quatro números que já vinham prontos, cinco que vieram de uma tradução de texto sem ordem inventada. `y` chega com as mesmas 10.692 linhas, numa coluna só: o que se quer prever. É a tabela que um modelo espera receber. O que falta agora é o que um modelo é, e como saber se ele presta.

## Leituras adicionais

*A escrever.*